In [ ]:
import os
cwd = os.path.normpath(os.getcwd())
cwd = cwd.split(os.sep)
find = cwd.index("fidelity-phase-tran")
newdir = f"{os.sep}".join(cwd[:find+1])
os.chdir(newdir)

import numpy as np
import pickle
import gzip

from qiskit.quantum_info import SparsePauliOp

from matplotlib import pyplot as plt

from qphaset.phases import gstates_to_rdms_matrix, phases_vfield
from qphaset.plotting import plot_grad_g_angle_stream

## Rank-1 solution for the order parameter discovery.

In [ ]:
filename = "annni_ext-c47e7c80-227f-4b18-b8a4-c910cc6d6908.pkl"            # ANNNI 50, c1=-1, upside down, floating detail, 64 x 64 (**)
# filename = "annni_ext-b9e5a93e-f105-411d-8013-b3120de2b88e.pkl"            # ANNNI 50, c1=-1, upside down, floating detail + multi-critical point, 64 x 64 (**)
# filename = 'results/ising-20spins-64x64-dmrg-20240214.pkl'

with gzip.open(filename, 'rb') as f:
    data = pickle.load(f)
params = data['params']
l, n = data['l'], data['n']
gstates = data['gstates']
stats = data['stats']

params_extent = np.concatenate([np.min(params, axis=0), np.max(params, axis=0)])
params_extent = tuple(params_extent[[0, 2, 1, 3]])

sites = [l // 2]
# sites = [l // 2, l // 2 + 1]

rdms = gstates_to_rdms_matrix(gstates, sites=sites)
rdms = rdms[::-1]   # TODO fix

In [ ]:
theta = np.pi  # Adjust st phases have opposite signs.
# Most of the times theta=0 is good, however, use theta=pi to obtain the complementary
# order parameter. 

In [ ]:
grad_g = phases_vfield(rdms, scale=1)
ys = np.sin(np.angle(grad_g) + theta)

# Labels plot
plt.matshow(np.sign(ys), origin='lower', extent=params_extent)

In [ ]:
rdms = rdms[1:-1, 1:-1] # TODO fix

In [ ]:
lattice_shape = rdms.shape[:2]
rdms = np.reshape(rdms, (-1, ) + rdms.shape[2:])
ys = ys.flatten()

In [ ]:
assert rdms.ndim == 3

# Normalization factors.
fp, fn = 1 / np.count_nonzero(ys > 0), 1 / np.count_nonzero(ys < 0)

gamma = 1
pos_y_term = -np.sum(rdms[np.nonzero(ys > 0)], axis=0) * fp
neg_y_term = np.sum(rdms[np.nonzero(ys < 0)], axis=0) * fn
m = pos_y_term + gamma * neg_y_term

assert np.allclose(m - m.conj(m).T, 0)
np.linalg.eigh(m)

m_eigval, m_eigvec = np.linalg.eigh(m)
eigval_min_abs_idx = np.argmin(np.abs(m_eigval))
v = m_eigvec[:, 0:1]
obs = v @ np.conj(v).T

In [ ]:
plt.matshow(obs)

In [ ]:
SparsePauliOp.from_operator(obs)

In [ ]:
meas = [np.trace(rdm @ obs) for rdm in rdms]
meas = np.reshape(meas, lattice_shape)

plt.matshow(meas, origin='lower', cmap='plasma', extent=params_extent)
plt.colorbar()

In [ ]:
plot_grad_g_angle_stream(grad_g, params_extent=params_extent, theory_lines=False);